In [ ]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
import seaborn as sns
import colorcet
import textalloc as ta

import itertools

from scipy import interpolate, stats, optimize
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import statsmodels.formula.api as smf
import mathieu as mh
import importlib

In [ ]:
importlib.reload(mh)

# Import metadata and define basic variables

In [ ]:
base_dir = '/home/mathieu/postdoc_heasley/experiments/growth_curves/HU_pheno/'
fig_path = '/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'

In [ ]:
experiments = pd.read_csv(f'{base_dir}script/experiments.csv').set_index('dataset')
exclude = pd.read_csv(f'{base_dir}script/exclude.csv').set_index('dataset')

In [ ]:
collection = pd.read_excel('/home/mathieu/postdoc_heasley/collection/mccusker_collection_wild_annot.xlsx')
collection.index = collection['ID'].values

spores_collection = pd.read_excel('/home/mathieu/postdoc_heasley/experiments/ho_deletion/spores_collection.xlsx')

strain_order = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]
strain_order = collection.loc[strain_order].sort_values(by='Y\' element').index
strain_color = dict(zip(strain_order, colorcet.glasbey_category10))
strain_order_rank = dict(zip(strain_order, range(10)))
#strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in collection.loc[strain_order, 'Y\' element']/150]))
strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in np.linspace(0,1,10)]))
hu_high_color = dict(zip([0]+[2**i for i in range(4,11)], [cm.magma(i) for i in np.linspace(0,1,8)]))

In [ ]:
class_color = dict(zip(['WT', 'HO hemizygote', 'HO null', 'MAT hemizygote', 'spore'], colorcet.glasbey_dark))
genotype_order = ['WT', 'HO/ho', 'ho/ho', 'ho/ho MATa', 'ho/ho MATx', 'ho MATa', 'ho MATx']
#genotype_color = dict(zip(genotype_order, ['0.2', '0.55', '0.9', '#3300FF', '#FFC400', '#B7A6FF', '#FFF0B5']))
genotype_color = dict(zip(genotype_order, ['0', '0.35', '0.7', '#3300FF', '#FFC400', '#3300FF', '#FFC400']))
genotype_color = dict(zip(genotype_order, ['0', '0.35', '0.7', 'dodgerblue', 'orange', 'dodgerblue', 'orange']))

genotype_alias = {'WT':'WT',
                 'HO/ho': r'$HO/ho\Delta$',
                 'ho/ho': r'$ho\Delta/ho\Delta$',
                 'ho/ho MATa': r'$ho\Delta/ho\Delta$'+'\n'+r'$MATa/mat\alpha\Delta$',
                 'ho/ho MATx': r'$ho\Delta/ho\Delta$'+'\n'+r'$mata\Delta/MAT\alpha$',
                 'ho MATa': r'$ho\Delta$ $MATa$',
                 'ho MATx': r'$ho\Delta$ $MAT\alpha$'}

In [ ]:
strains_info = pd.read_csv(f'{base_dir}script/strains_info.csv').set_index('strain')

In [ ]:
for (mat, ho), df in strains_info.groupby(['mating_type', 'HO']):
    if mat == 'a/x':
        if ho == 'HO/HO':
            c = 'WT'
        elif ho in ['HO/ho::hphMX', 'HO/ho::natMX']:
            c = 'HO hemizygote'
        elif ho in ['ho::hphMX/ho::natMX', 'ho::natMX/ho::hphMX']:
            c = 'HO null'
    elif mat in ['a', 'x']:
        if ho in ['ho::hphMX', 'ho::natMX']:
            c = 'spore'
        elif ho in ['ho::hphMX/ho::natMX', 'ho::natMX/ho::hphMX']:
            c = 'MAT hemizygote'
    strains_info.loc[df.index, 'class'] = c

In [ ]:
wine_subclade = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]

# Growth curves analysis
The parse_gc function needs two file paths as input: 

1) the raw plate reader text file
2) a standardized tab file with samples definition, in long-form format

The output is two objects:
1) the samples specification sheet, to be stored in a dictionnary for future use by other functions
2) a long-form table of all OD values with metadata

The experiments object is a table describing the distinct plate reader runs to be combined in a single analysis. The loop in the next cell iterates trough each run to run parse_gc individually, and then creates a concatenated DataFrame object that comprises all run

In [ ]:
GC_DAT = []
GC_samples = {}

for ds, (dat_file, sm_file, temp, media, duration, blank_correct) in experiments.iterrows():

    dat_path = f'{base_dir}data/{dat_file}'
    sm_path = f'{base_dir}script/{sm_file}'
        
    sm, gc_dat = mh.parse_gc(dat_path, sm_path, ds, skiprows=11, blank_correct=blank_correct, Tend=duration)
    
    GC_DAT.append(gc_dat)
    GC_samples[ds] = sm
GC_DAT = pd.concat(GC_DAT).reset_index(drop=True)
GC_DAT = GC_DAT.loc[(~GC_DAT['OD600'].isna()) & (GC_DAT['strain']!='blank')]
GC_DAT = GC_DAT.merge(strains_info, on='strain')
# exclude ugly wells
for (ds, w), df in GC_DAT.groupby(['dataset', 'well']):
    if ds in exclude.index:
        if w in exclude.loc[ds, ['well']].values:
            GC_DAT.drop(df.index, inplace=True)

for (ds, w), df in GC_DAT.groupby(['dataset', 'well']):
    zero = df.set_index('time').loc[0, 'OD600_blank']
    GC_DAT.loc[df.index, 'log_OD600'] = np.log(df['OD600_blank']/zero)

## Per-well inspection

In [ ]:
# per-well inspection

ds_color_dict = dict(zip(GC_DAT['dataset'].unique(), colorcet.glasbey_bw))

for well, df in GC_DAT.groupby('well'):

    if well == 'M22':

        fig, ax = plt.subplots(figsize=[5,5])
    
        for ds, df1 in df.groupby('dataset'):
            ax.plot(df1['time'], df1['OD600'], c=ds_color_dict[ds], label=ds)
            
        ax.set_xticks(np.arange(0, 30, 4), minor=False)
        ax.set_xticks(np.arange(0, 30, 1), minor=True)
        ax.grid(which='major', lw=1)
        ax.grid(which='minor', lw=0.3)
    
        ax.legend(loc=2)
        
        ax.set_ylim(0, 2.6)
        ax.set_xlabel('hrs')
        ax.set_ylabel('OD$_{600}$')
        ax.set_title(well)
    
        plt.tight_layout()
        #plt.savefig(f'{fig_path}per_well/{well}.png', dpi=300)
        plt.show()
        plt.close()

# Compute AUC
The compute_gc_auc function takes as input both outputs from parse_gc and produces two outputs:
1) AUC values for all non-blank positions in the plate, one curve per well
2) AUC values per genotype x condition (or any other metadata variables to be kept in the final table), averaging technical replicates

In [ ]:
GC_AUC = []
GC_AUC_MEAN = []
average_reps = ['medium', 'phase']

for ds, gc_dat in GC_DAT.groupby('dataset'):
    
    gc_samples = GC_samples[ds]
    gc_auc, gc_auc_mean = mh.compute_gc_auc(gc_dat, GC_samples[ds], strains_info, ds, od='OD600', 
                                            average_reps=['strain', 'HU']+[c for c in average_reps if c in gc_samples.columns])
    GC_AUC.append(gc_auc)
    GC_AUC_MEAN.append(gc_auc_mean.merge(strains_info, left_on='strain', right_on='strain'))

GC_AUC = pd.concat(GC_AUC).reset_index(drop=True)
GC_AUC = GC_AUC.loc[GC_AUC['strain']!='blank']

GC_AUC_MEAN = pd.concat(GC_AUC_MEAN).reset_index(drop=True)
GC_AUC_MEAN = GC_AUC_MEAN.loc[GC_AUC_MEAN['strain']!='blank']

for (ds, s, m, p), df in GC_AUC.groupby(['dataset', 'strain', 'medium', 'phase']):
    if 0 in df['HU'].values:
        GC_AUC.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['auc']).values

for (ds, s, m, p), df in GC_AUC_MEAN.groupby(['dataset', 'strain', 'medium', 'phase']):
    if 0 in df['HU'].values:
        GC_AUC_MEAN.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['mean_auc']).values

# Compute logistic fit

In [ ]:
GC_LOGISTIC = []
GC_LOGISTIC_MEAN = []
average_reps = ['medium', 'phase']

for od in ['OD600_blank', 'log_OD600']:
    if od == 'OD600_blank':
        bounds = (-np.inf, np.inf)
    elif od == 'log_OD600':
        bounds = ((0, 0, -np.inf), (np.inf, np.inf, np.inf))
    else:
        bounds = np.nan
    
    for ds, gc_dat in GC_DAT.groupby('dataset'):

        if experiments.loc[ds, 'blank'] == True:
    
            gc_samples = GC_samples[ds]
            gc_logistic, gc_logistic_mean = mh.compute_gc_logistic(gc_dat, gc_samples, strains_info, ds, od=od, bounds=bounds,
                                                                   average_reps=['strain', 'HU']+[c for c in average_reps if c in gc_samples.columns])
            gc_logistic['OD'] = od
            gc_logistic_mean['OD'] = od
            
            GC_LOGISTIC.append(gc_logistic)
            GC_LOGISTIC_MEAN.append(gc_logistic_mean.merge(strains_info, left_on='strain', right_on='strain'))

GC_LOGISTIC = pd.concat(GC_LOGISTIC).reset_index(drop=True)
GC_LOGISTIC_MEAN = pd.concat(GC_LOGISTIC_MEAN).reset_index(drop=True)

# Add inhibition to GC_LOGISTIC
for (ds, od, s, m, p), df in GC_LOGISTIC.groupby(['dataset', 'OD', 'strain', 'medium', 'phase']):
    if 0 in df['HU'].values:
        GC_LOGISTIC.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['mu']).values

# Add inhibition to GC_LOGISTIC_MEAN

for (ds, od, s, m, p), df in GC_LOGISTIC_MEAN.groupby(['dataset', 'OD', 'strain', 'medium', 'phase']):
    if 0 in df['HU'].values:
        GC_LOGISTIC_MEAN.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['mean_mu']).values

In [ ]:
#GC_LOGISTIC.to_csv(f'{base_dir}tables/GC_LOGISTIC.csv', index=False)
GC_LOGISTIC = pd.read_csv(f'{base_dir}tables/GC_LOGISTIC.csv')

# Fitness defect

In [ ]:
gclog = GC_LOGISTIC.loc[(GC_LOGISTIC['background'].isin(strain_order))
    & (GC_LOGISTIC['HU']==0)
    & (GC_LOGISTIC['OD']=='log_OD600')].set_index(['dataset','medium','phase','strain']).copy()
gclog = gclog.loc[('wt_log_sat', 'unknown', 'sat')].sort_values(by='mu')
gclog['N24'] = gclog['mu'].apply(lambda m: np.e**(m*24))
gclog['N24*'] = gclog['mu'].apply(lambda m: 2**(24/((np.log(2)/m))))
gclog['s'] = np.log2(gclog['N24']/gclog.loc['YJM967', 'N24'])/24
gclog['s*'] = np.log2(gclog['N24*']/gclog.loc['YJM967', 'N24*'])/24
gclog['mu_s'] = (gclog['mu']/gclog.loc['YJM967', 'mu'])-1

## s calculation

In [ ]:
yprime_kb_s = {}
rDNA_cn_s = {}

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['background'].isin(strain_order))
    & (GC_LOGISTIC['OD']=='log_OD600')
    & (GC_LOGISTIC['HU']==0)].set_index(['dataset','medium','phase']).sort_index()


for cond, idx in zip(['YPD', 'SC'], 
                     [('wt_ypd_sc', 'YPD', 'unknown'), ('wt_log_sat', 'unknown', 'sat')]):
    
    df = dat.loc[idx].sort_values(by='yprime_kb')
    X = df['yprime_kb'].iloc[[0, -1]].values
    lr = stats.linregress(df['yprime_kb'], df['mu'])
    Y = X*lr.slope+lr.intercept
    s = (Y[1]/Y[0])-1
    yprime_kb_s[cond] = s

    df = dat.loc[idx].sort_values(by='rDNA_cn')
    X = df['rDNA_cn'].iloc[[0, -1]].values
    lr = stats.linregress(df['rDNA_cn'], df['mu'])
    Y = X*lr.slope+lr.intercept
    s = (Y[1]/Y[0])-1
    rDNA_cn_s[cond] = s

## Fig 3A

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=[8,3],
                        gridspec_kw=dict(wspace=0.3, 
                                         left=0.12, top=0.9, right=0.84, bottom=0.15))

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['background'].isin(strain_order))
    & (GC_LOGISTIC['OD']=='OD600_blank')
    & (GC_LOGISTIC['HU']==0)].set_index(['dataset','medium','phase','background'])

legend_elms = []

for cond, idx, ax_idx in zip(['YPD', 'SC'], 
                             [('wt_ypd_sc', 'YPD', 'unknown'), ('wt_log_sat', 'unknown', 'sat')],
                            [0,1]):

    ax = axes[ax_idx]
    s = f'{yprime_kb_s[cond]:.3f}'.replace('-', '\N{MINUS SIGN}')
    
    
    marker = ('s' if cond == 'YPD' else '^' if cond == 'SC' else None)
    df = dat.loc[idx].loc[strain_order]
    ax.scatter(df['yprime_kb'], df['mu'], 
               #marker=marker,
               c=[strain_yprime_color[s] for s in strain_order],
              zorder=1)
    
    X = df['yprime_kb'].iloc[[0, -1]].values
    lr = stats.linregress(df['yprime_kb'], df['mu'])
    Y = X*lr.slope+lr.intercept
    
    ax.plot(X, Y, c='0.5', lw=1, zorder=0)
    #ax.plot([1450, 1500, 1500, 1450], np.repeat(Y, 2), c='0.5', lw=1)
    ax.text(0.9, 0.9, f's={s}\np={mh.plot_pval_text(lr.pvalue)}',
           color='0.5', va='top', ha='right', transform=ax.transAxes)

    ax.set_xticks(np.linspace(0, 1500, 4))
    ax.set_xlabel('Y\' kb')
    ax.set_ylabel('OD600 hr$^{-1}$')
    ax.set_title(cond)

legend_elms = [Line2D([0], [0], c='w', marker='o', ms=10,
                      label=s, 
                     mfc=strain_yprime_color[s])
              for s in strain_order]
ax.legend(handles=legend_elms, loc=6, frameon=False, bbox_to_anchor=[1, 0.5])

fig.text(0.02, 0.91, 'A', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig3A.{ext}', dpi=300)

plt.show()
plt.close()

## Fig 3C

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=[8,3],
                        gridspec_kw=dict(wspace=0.3, 
                                         left=0.12, top=0.9, right=0.84, bottom=0.15))

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['background'].isin(strain_order))
    & (GC_LOGISTIC['OD']=='OD600_blank')
    & (GC_LOGISTIC['HU']==0)].set_index(['dataset','medium','phase','background'])

legend_elms = []

for cond, idx, ax_idx in zip(['YPD', 'SC'], 
                             [('wt_ypd_sc', 'YPD', 'unknown'), ('wt_log_sat', 'unknown', 'sat')],
                            [0,1]):

    ax = axes[ax_idx]
    s = f'{rDNA_cn_s[cond]:.3f}'.replace('-', '\N{MINUS SIGN}')
    
    
    marker = ('s' if cond == 'YPD' else '^' if cond == 'SC' else None)
    df = dat.loc[idx].sort_values(by='rDNA_cn')
    ax.scatter(df['rDNA_cn'], df['mu'], 
               #marker=marker,
               c=[strain_yprime_color[s] for s in df.index],
              zorder=1)
    
    X = df['rDNA_cn'].iloc[[0, -1]].values
    lr = stats.linregress(df['rDNA_cn'], df['mu'])
    Y = X*lr.slope+lr.intercept
    
    ax.plot(X, Y, c='0.5', lw=1, zorder=0)
    #ax.plot([1450, 1500, 1500, 1450], np.repeat(Y, 2), c='0.5', lw=1)
    ax.text(0.9, 0.9, f's={s}\np={mh.plot_pval_text(lr.pvalue)}',
           color='0.5', va='top', ha='right', transform=ax.transAxes)

    #ax.set_xticks(np.linspace(0, 1500, 4))
    ax.set_xlabel('rDNA CN')
    ax.set_ylabel('OD600 hr$^{-1}$')
    ax.set_title(cond)

legend_elms = [Line2D([0], [0], c='w', marker='o', ms=10,
                      label=s, 
                     mfc=strain_yprime_color[s])
              for s in strain_order]
ax.legend(handles=legend_elms, loc=6, frameon=False, bbox_to_anchor=[1, 0.5])

fig.text(0.02, 0.91, 'C', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig3C.{ext}', dpi=300)

sns.despine()

plt.show()
plt.close()

In [ ]:
dat = strains_info.loc[wine_subclade]
stats.pearsonr(dat['rDNA_cn'], dat['yprime_kb'])

# IC50

In [ ]:
GC_IC50 = []
for ds, df in GC_LOGISTIC.loc[(GC_LOGISTIC['OD']=='OD600_blank') & (~GC_LOGISTIC['inhibition'].isna())].groupby('dataset'):
#for ds, df in GC_AUC.loc[(~GC_AUC['inhibition'].isna())].groupby('dataset'):
    if ds in ['all_genotypes', 'pseudohap_batch1', 'pseudohap_batch2', 'pseudohap_batch3', 'pseudohap_batch4', 'wt_ypd_sc', 'wt_log_sat']:
        gc_ic50 = []
        for (s, m, p), df1 in df.groupby(['strain', 'medium', 'phase']):
            if df1.shape[0] > 1:
                try:
                    (ic50, hill_coeff), cov = optimize.curve_fit(mh.hill_equation, df1['HU'], df1['inhibition'])
                except RuntimeError:
                    (ic50, hill_coeff) = (np.nan, np.nan)
            else:
                (ic50, hill_coeff) = (np.nan, np.nan)
                
            gc_ic50.append([s, ds, m, p, ic50, hill_coeff])
        
        gc_ic50 = pd.DataFrame(gc_ic50, columns=['strain', 'dataset', 'medium', 'phase', 'IC50', 'hill_coeff'])
    
        GC_IC50.append(gc_ic50)

GC_IC50 = pd.concat(GC_IC50).reset_index(drop=True)
GC_IC50 = GC_IC50.merge(strains_info, on='strain')

In [ ]:
GC_IC50['IC25'] = GC_IC50.apply(lambda x: mh.hill_equation_reciprocal(0.25, x['IC50'], x['hill_coeff']), axis=1)
GC_IC50['IC75'] = GC_IC50.apply(lambda x: mh.hill_equation_reciprocal(0.75, x['IC50'], x['hill_coeff']), axis=1)

In [ ]:
#GC_IC50.to_csv(f'{base_dir}tables/GC_IC50.csv', index=False)
GC_IC50 = pd.read_csv(f'{base_dir}tables/GC_IC50.csv')

## Plot fits of IC50

In [ ]:
#for (s, m, p), df in GC_AUC.loc[GC_AUC['dataset'].isin(['all_genptypes', 'wt_log_sat'])].groupby(['strain', 'medium', 'phase']):
for (s, m, p), df in GC_LOGISTIC.loc[GC_LOGISTIC['dataset'].isin(['all_genptypes', 'wt_log_sat'])].groupby(['strain', 'medium', 'phase']):
    fig, ax = plt.subplots(figsize=[5,5])
    for ds, df1 in df.groupby('dataset'):

        c=ds_color_dict[ds]
        ic50, hill_coeff = GC_IC50.set_index(['strain', 'medium', 'phase', 'dataset']).loc[(s, m, p, ds), ['IC50', 'hill_coeff']]
     
        ax.scatter(df1['HU'], df1['inhibition'], color=c, label=ds)
        X = np.linspace(1, 1024, 40)
        Y = np.apply_along_axis(lambda x: mh.hill_equation(x, ic50, hill_coeff), 0, X)
        ax.plot(X, Y, c=c)
    ax.legend()
    ax.set_title(f'{s} {m} {p}')
    plt.show()
    plt.close()

## Fit IC25 and IC75, and viz the closest matching growth curves from the latest run.

In [ ]:
S = ['YJM947', 'YJM955', 'YJM964', 'YJM967']
ax_idx = dict(zip(S, itertools.product((0,1), (0,1))))
ic50_sub = GC_IC50.loc[(GC_IC50['dataset']=='wt_log_sat') & (GC_IC50['phase']=='sat')].set_index('strain').loc[S]
dat_sub = GC_DAT.loc[(GC_DAT['dataset']=='wt_log_sat') & (GC_DAT['phase']=='sat')].set_index(['strain', 'HU']).loc[S]

fig, axes = plt.subplots(ncols=2, nrows=2, figsize=[11,8])

for s in S:

    ax = axes[ax_idx[s]]

    dat = dat_sub.loc[s].loc[0].sort_values(by='time')
    ax.plot(dat['time'], dat['OD600'], c='k', label='0 mM')
    
    ic50, hill_coeff = ic50_sub.loc[s, ['IC50', 'hill_coeff']]
    Y = [0.25, 0.5, 0.75]
    C = np.round(np.apply_along_axis(lambda y: mh.hill_equation_reciprocal(y, ic50, hill_coeff), 0, Y))
    
    for i,c,y in zip([1.3, 1.2, 1.1], C, Y):
        closest = dat_sub.reset_index()['HU'].unique()
        closest = closest[closest>0]
        closest = (pd.Series(np.log2(closest), index=closest)-np.log2(c)).abs().sort_values().index[0]

        dat = dat_sub.loc[s].loc[closest].sort_values(by='time')
        ax.plot(dat['time'], dat['OD600'], label=f'{closest} mM')
        ax.text(0, i, f'IC{y*100:.0f}: {c:.0f} mM')
    
    ax.axhline(0.5, ls='--', c='0.5')

    ax.set_ylim(0, 1.8)
    ax.set_ylabel('OD600')
    ax.set_xticks(range(25), minor=True)
    ax.set_xlabel('hrs')

    ax.legend(loc=4)

    ax.set_title(s)

plt.tight_layout()
#plt.savefig(f'{fig_path}/IC25_IC50_IC75_curves.png', dpi=300)
plt.show()
plt.close()

## Fig 5B

In [ ]:
fig = plt.figure(figsize=[7, 3])
gs = plt.GridSpec(ncols=2, nrows=1, wspace=0.35, 
                    left=0.15, right=0.95, top=0.85, bottom=0.25)

C = [0, 133]
S = strain_order
S = [strain_order[1]] + list(strain_order.delete(1))

X = np.array([30, 1400])


dat = GC_LOGISTIC.loc[(GC_LOGISTIC['dataset']=='wt_log_sat') &
    (GC_LOGISTIC['phase']=='sat') &
    (GC_LOGISTIC['OD']=='OD600_blank') &
    (GC_LOGISTIC['strain'].isin(wine_subclade))].copy().set_index('HU')
dat['YJM948'] = dat['background'] == 'YJM948'

for ax_idx, conc in zip([0,1], C):
    ax = fig.add_subplot(gs[ax_idx])

    df = dat.loc[conc]

    sns.barplot(x='strain', y='mu', order=S,
                hue='strain', palette=strain_yprime_color,
                saturation=1,
               data=dat.loc[conc], ax=ax)

    if conc == 0:
        ax.set_title('SC')
    else:
        ax.set_title(f'HU {conc} mM')
    
    #ax.set_xlabel('Y\' kb')
    ax.set_ylabel(r'OD$_{600}$ hr$^{-1}$ ')
    #ax.set_xlim(-250, 1600)
    ax.set_ylim(0, 0.3)
    ax.tick_params(axis='x', rotation=90)
    ax.set_xlabel('')


fig.text(0.02, 0.9, 'B', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig5B.{ext}', dpi=300)

plt.show()
plt.close()

## Fig 5C-E

In [ ]:
fig = plt.figure(figsize=[10,3.5])
gs = plt.GridSpec(ncols=3, nrows=1, wspace=0.35,
                    left=0.08, right=0.88, top=0.85, bottom=0.22)

S = [strain_order[1]] + list(strain_order.delete(1))
strain_marker = {s:('^' if s=='YJM948' else 'o') for s in S}

## Dose-response curve

ax = fig.add_subplot(gs[0])

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['dataset']=='wt_log_sat') &
    (GC_LOGISTIC['phase']=='sat') &
    (GC_LOGISTIC['OD']=='OD600_blank') &
    (GC_LOGISTIC['strain'].isin(wine_subclade))].copy()
dat['log_hu'] = np.log10(dat['HU'].replace({0:3}))


X_lin = np.sort(dat['HU'].unique())
X_lin_small = np.linspace(X_lin[0], X_lin[-1], 100)
X_log = np.sort(dat['log_hu'].unique())
X_log_small = X_lin_small
X_log_small[X_log_small==0] = 3
X_log_small = np.log10(X_log_small)

sns.scatterplot(x='log_hu', y='inhibition', 
                hue='background', palette=strain_yprime_color,
                data=dat, ax=ax, legend=False)

dat =  GC_IC50.loc[(GC_IC50['dataset']=='wt_log_sat') &
    (GC_IC50['phase']=='sat') &
    (GC_IC50['strain'].isin(wine_subclade))].copy()#.set_index('strain')
dat['YJM948'] = dat['background'] == 'YJM948'
# Add extra kb estimate
dat['extra_kb'] = np.where(dat['strain']=='YJM948', dat['yprime_kb']+563, dat['yprime_kb'])

for i, (s, ic50, hill) in dat[['strain', 'IC50', 'hill_coeff']].iterrows():
    c = palette=strain_yprime_color[s]
    Y = np.apply_along_axis(lambda x: mh.hill_equation(x, ic50, hill), 0, X_lin_small)
    ax.plot(X_log_small, Y, color=c)


Y = np.linspace(0,1,6)
ax.set_yticks(Y, labels=(Y*100).astype(int))
ax.set_ylabel('inhibition (%)')
ax.set_xticks(X_log, labels=X_lin, rotation=90)
ax.set_xlabel('HU (mM)')

## IC50 values

ax = fig.add_subplot(gs[1])
X = np.arange(len(S))
ax.bar(X, dat.set_index('strain').loc[S, 'IC50'], color=[strain_yprime_color[s] for s in S])
ax.set_xticks(X, labels=S, rotation=90)
ax.set_ylabel('IC50 (mM)')

## Correlation of IC50 with extra kb

ax = fig.add_subplot(gs[2])

X = np.array([30, 1400])

for i, (s, x, y) in dat[['strain', 'extra_kb', 'IC50']].iterrows():
    ax.scatter(x, y, marker=strain_marker[s], color=strain_yprime_color[s], 
              zorder=2)

lm = stats.linregress(dat['extra_kb'], dat['IC50'])
ax.plot(X, X*lm.slope+lm.intercept, c='0.5', lw=1, zorder=1)
ax.text(0.9, 0.9, 'r$^2$'+f'={lm.rvalue**2:.2f}\np={mh.plot_pval_text(lm.pvalue)}', 
        ha='right', va='top', c='0.5', transform=ax.transAxes)

ax.set_xlim(-250, 1600)
ax.set_xlabel('extra kb')
ax.set_ylabel('IC50 (mM)')

ax = fig.add_subplot(gs[:])
ax.axis('off')

legend_elms = [Line2D([0], [0], c='w', marker=strain_marker[s], ms=10,
                      label=s, 
                     mfc=strain_yprime_color[s])
              for s in strain_order]
ax.legend(handles=legend_elms, loc=6, frameon=False, bbox_to_anchor=[1, 0.5])

fig.text(0.02, 0.91, 'C', size=24, fontweight='bold')
fig.text(0.32, 0.91, 'D', size=24, fontweight='bold')
fig.text(0.61, 0.91, 'E', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig5C-E.{ext}', dpi=300)

plt.show()
plt.close()

## Fig S7

In [ ]:
fig = plt.figure(figsize=[8,4])
gs = plt.GridSpec(ncols=1, nrows=1, 
                  left=0.08, right=0.8, top=0.93, bottom=0.2)
ax = fig.add_subplot(gs[0])

S = [f'YJM{i}' for i in (947, 948, 954, 956, 957, 963, 964, 965)]
S = [s for s in strain_order if s in S]
S = strain_order

dat = GC_LOGISTIC_MEAN.loc[(GC_LOGISTIC_MEAN['dataset']=='pseudohap_batch5') 
    & (GC_LOGISTIC_MEAN['OD']=='OD600_blank')
    & (GC_LOGISTIC_MEAN['background'].isin(S))
    & (GC_LOGISTIC_MEAN['HU']==128)
    ].copy()
dat['inhibition'] *= 100

# on-the-spot normalization for IC50 of WT

for bg, df in dat.groupby('background'):
    wt_value = df.set_index('genotype').loc['WT', 'inhibition'].mean()
    dat.loc[df.index, 'inhib_ratio'] = df['inhibition']/wt_value
dat['inhib_ratio_pseudo1'] = dat['inhib_ratio']-1

sns.barplot(x='background', y='inhibition', order=S, errorbar=None, 
             hue='genotype', palette=genotype_color, hue_order=genotype_order,
              data=dat, zorder=2, legend=False, saturation=1,
           ax=ax)
sns.stripplot(x='background', y='inhibition', order=S, 
              size=6, edgecolor='w', linewidth=1, dodge=True, 
              hue='genotype', palette=genotype_color, hue_order=genotype_order,
              data=dat, zorder=2, legend=False,
             ax=ax)


ax.set_ylabel('HU 128 mM inhibition (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=90)

legend_elms = [Line2D([0], [0], c='w', marker='o', ms=10,
                      label=genotype_alias[gt],
                     mfc=genotype_color[gt])
              for gt in genotype_order[:-2]]
ax.legend(handles=legend_elms, loc=6, frameon=False, bbox_to_anchor=[1, 0.5])

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS7.{ext}', dpi=300)
    
plt.show()
plt.close()

## Fig 5F-G

In [ ]:
fig = plt.figure(figsize=[10, 3.5])
gs = plt.GridSpec(ncols=2, nrows=1, wspace=0.3, 
                 left=0.08, right=0.85, top=0.87, bottom=0.19)

S = [strain_order[1]] + list(strain_order.delete(1))

ax = fig.add_subplot(gs[0])

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['dataset'].isin(['spores_batch1','spores_batch2']))
    & (GC_LOGISTIC['OD']=='OD600_blank')
    & (GC_LOGISTIC['background'].isin(wine_subclade))
    & (GC_LOGISTIC['HU']==128)
    & (~GC_LOGISTIC['inhibition'].isna())].groupby('strain')['inhibition']\
    .mean().reset_index().merge(strains_info, left_on='strain', right_on='strain')

X = np.arange(len(strain_order))
sns.boxplot(x='background', y='inhibition', order=S,
           color='k', fill=False, linewidth=1,
            fliersize=0,
            data=dat.loc[dat['genotype']!='WT'], ax=ax, zorder=1)
sns.stripplot(x='background', y='inhibition', order=S,
              s=4, edgecolor='k', linewidth=0.5,
            hue='background', palette=strain_yprime_color,
            data=dat.loc[dat['genotype']!='WT'], ax=ax, zorder=2)

ax.bar(np.arange(10), dat.set_index('strain').loc[S, 'inhibition'], 
       width=1, color=[strain_yprime_color[s] for s in S], alpha=0.7, zorder=0)


for x, s in enumerate(S):

    n_spores = dat.set_index('background').loc[s].shape[0]-1
    ax.text(x, 1.05, f'({n_spores})', size=9, ha='center', va='center')


Y = np.linspace(0,1,6)
ax.set_yticks(Y, labels=(Y*100).astype(int))
ax.set_ylabel('HU 128 mM inhibition (%)')

ax.set_xlabel('')
ax.tick_params(axis='x', rotation=90)

## Spores correlation with CN

ax = fig.add_subplot(gs[1])
X = np.array([0, 225])
gt_z_order = {'ho MATa':1, 'ho MATx':1, 'WT':2}

dat = dat.loc[(~dat['yprime_cn'].isna())
    & ((dat['chr8_cn']==1) | (dat['genotype']=='WT'))
    & (dat['strain']!='YJM948')].set_index('genotype')
legend_elms = []

for gt in ['WT', 'ho MATa', 'ho MATx']:
    df = dat.loc[gt]
    c = genotype_color[gt]
    z = gt_z_order[gt]
    ax.scatter(df['yprime_cn'], df['inhibition'], 
               s=12, c=c, zorder=z)
    lm = stats.linregress(df['yprime_cn'], df['inhibition'])

    Y = X*lm.slope+lm.intercept
    ax.plot(X, Y, ls='-', lw=2, color=c, zorder=z)

    legend_elms.append(Line2D([0], [0], c='w', marker='o', ms=8, mfc=genotype_color[gt],
                              label=f'{genotype_alias[gt]}\nr$^2$={mh.plot_pval_text(lm.rvalue**2)}\np={mh.plot_pval_text(lm.pvalue)}'))

Y = np.linspace(0,1,6)
ax.set_yticks(Y, labels=(Y*100).astype(int))
ax.set_ylabel('HU 128 mM inhibition (%)')
ax.set_xlabel('Y\' CN')


ax.legend(handles=legend_elms, loc=6, frameon=False, bbox_to_anchor=[1, 0.5], labelspacing=2)


fig.text(0.02, 0.91, 'F', size=24, fontweight='bold')
fig.text(0.44, 0.91, 'G', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig5FG.{ext}', dpi=300)
plt.show()
plt.close()

# Spores

In [ ]:
dat = GC_LOGISTIC_MEAN.loc[GC_LOGISTIC_MEAN['dataset'].isin(['spores_batch1', 'spores_batch2']) & 
    (GC_LOGISTIC_MEAN['background']!='YJM434') & (GC_LOGISTIC_MEAN['HU']==128) & 
    (~GC_LOGISTIC_MEAN['inhibition'].isna())].copy()
dat['strain'] = dat['strain'].apply(lambda x: x.replace(' ', '_'))
dat.to_csv('/home/mathieu/postdoc_heasley/wine_subclade_spores/tables/hu_pheno.csv', index=False)

## Between-run variability in growth curves

In [ ]:
runs_palette = dict(zip(['all_genotypes', 'spores_batch1', 'spores_batch2', 'pseudohap_batch1'], colorcet.glasbey_bw))

for s, df in GC_DAT.loc[GC_DAT['HU'].isin([0, 128])].groupby('strain'):
    if s in strain_order:
        fig, ax = plt.subplots()
        ax.set_title(s)

        sns.lineplot(x='time', y='OD600', hue='dataset', style='HU', data=df,
                    palette=runs_palette, units='well', estimator=None, ax=ax)

In [ ]:
dat = GC_IC50.loc[(GC_IC50['ho']=='HO') & (GC_IC50['dataset'].isin(['all_genotypes', 'pseudohap_batch1']))]
fig, axes = plt.subplots(ncols=3, figsize=[14,5])
ax = axes[0]
sns.barplot(x='background', y='IC50', hue='dataset', order=strain_order, data=dat, ax=ax)
ax.tick_params(axis='x', labelsize=9, rotation=90)

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['ho']=='HO') &
    (GC_LOGISTIC['dataset'].isin(['all_genotypes', 'pseudohap_batch1'])) & 
    (GC_LOGISTIC['background']!='YJM434')]

for ds, ax_idx in zip(['all_genotypes', 'pseudohap_batch1'], (1,2)):
    ax = axes[ax_idx]
    dat1 = dat.set_index('dataset').loc[ds]
    sns.lineplot(x='HU', y='inhibition', hue='background', hue_order=strain_order, palette=strain_color, data=dat1, ax=ax)
    ax.set_title(ds)

In [ ]:
dat = GC_IC50.loc[(GC_IC50['background'].isin(['YJM947', 'YJM948', 'YJM954', 'YJM956', 'YJM963', 'YJM964'])) &
    (GC_IC50['dataset'].isin(['pseudohap_batch1','pseudohap_batch2']))]
mat_palette = {'a/x':'k', 'a':'r', 'x':'b'}
for bg, df in dat.groupby('background'):

    S = list(df['strain'])
    S.remove(bg)
    S = [bg] + S
    fig, ax = plt.subplots(figsize=[5,5])
    
    sns.barplot(x='strain', y='IC50', order=S, hue='mating_type', palette=mat_palette, data=df, ax=ax)
    ax.tick_params(axis='x', labelsize=9, rotation=90)
    ax.set_ylim(0, 120)
    
    plt.show()
    plt.close()
    

# Selection of spores for CRISPR

In [ ]:
spores_info = strains_info.loc[(strains_info['class']=='spore') & (~strains_info['yprime_cn'].isna())].copy()
spores_info['parent_yprime_cn'] = strains_info.loc[strains_info['class']=='WT', 'yprime_cn'].loc[spores_info['background']].values
spores_info['delta_yprime_cn'] = np.abs(spores_info['yprime_cn']-spores_info['parent_yprime_cn'])

In [ ]:
spores_info.sort_values(by='delta_yprime_cn').reset_index().set_index(['background','strain'])[['MAT','yprime_cn','parent_yprime_cn','delta_yprime_cn',]]#.loc['YJM967']